# 1 — Ingesta Bronze incremental

`COPY INTO` registra qué archivos físicos ya procesó. Volver a ejecutar esta tarea no vuelve a cargar el mismo CSV. Luego se expone una vista unificada entre el Bronze inicial y los lotes nuevos.

In [ ]:
dbutils.widgets.text("student_id", "alumno01")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"])
dbutils.widgets.text("expected_batch_id", "batch_002")
dbutils.widgets.text("job_run_id", "interactive")

In [ ]:
%run ../common/config

In [ ]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
initial = qualified_table(config, "bronze_transactions")
incremental = qualified_table(config, "bronze_transactions_incremental")
unified = qualified_table(config, "bronze_transactions_all")
assert spark.catalog.tableExists(initial), "Falta bronze_transactions. Ejecutá la práctica 1."

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {incremental} (
  transaction_id STRING, customer_id STRING, product_id STRING, event_ts STRING,
  amount STRING, payment_channel STRING, device_id STRING, is_fraud STRING,
  source_batch_id STRING, updated_at STRING
) USING DELTA
""")
copy_result = spark.sql(f"""
COPY INTO {incremental}
FROM '{config.volume_path}/incoming/transactions/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'mode' = 'PERMISSIVE')
""")
display(copy_result)

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {unified} AS
SELECT transaction_id, customer_id, product_id, event_ts, amount, payment_channel, device_id, is_fraud,
       'initial' AS source_batch_id, event_ts AS updated_at
FROM {initial}
UNION ALL
SELECT transaction_id, customer_id, product_id, event_ts, amount, payment_channel, device_id, is_fraud,
       source_batch_id, updated_at
FROM {incremental}
""")
# expected_batch_id se comprueba en 04_validate_pipeline; la ingesta procesa todos los archivos nuevos.
display(spark.table(incremental).groupBy("source_batch_id").count().orderBy("source_batch_id"))